# Financial Sentiment Classification with BERT Fine-Tuning
**Cameron Batts**

Python · NLP · Transformers · PyTorch · BERT · FinBERT · scikit-learn

## Overview
Compare bag-of-words classifiers, an off-the-shelf sentiment checkpoint, head-only BERT adaptation and full BERT fine-tuning on financial-news sentiment. Inspect an external FinBERT model separately because its training data may overlap the evaluation corpus.

## Business context
Classifying financial text can help prioritize news for review. Model selection should account for minority-class recall, domain adaptation, compute cost and evaluation independence. This experiment does not measure investment returns or deployed business impact.

## Data and attribution
The supplied Financial PhraseBank-style CSV contains 4,846 labeled sentences. Source: Malo et al. (2014), *Good debt or bad debt: Detecting semantic orientations in economic texts*, https://doi.org/10.1002/asi.23062 . A 50% random sample yields 2,423 sentences. Raw text and saved person-name examples are omitted from this package. Place a permitted copy of `all-data.csv` under `data/` before running.

## Execution status
Selected classification reports are retained from the original completed notebook and rounded to two decimals. The cleaned workflow is syntax-checked, not rerun. Training logs, local paths and saved data excerpts are removed; model checkpoints are not bundled.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn import naive_bayes
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
news_df = pd.read_csv(root / 'data' / 'all-data.csv', names=['sentiment', 'content'], encoding='latin-1')
news_df = news_df.sample(frac=0.5, random_state=41)
print(news_df['sentiment'].value_counts())


## Train, validation and test split
Use the original random_state=41 split: 1,453 training, 485 validation and 485 test sentences. The split is not stratified and does not deduplicate related sentences. Test support is 51 negative, 297 neutral and 137 positive. A neutral-only prediction would therefore have about 61.2% accuracy, making class-level and macro metrics essential.

In [ ]:

X = news_df["content"]
y = news_df["sentiment"]

# First split: 60% train, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=41)

# Second split: cut the 40% temp in half, so 20% validation and 20% test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=41)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))


## Bag-of-words benchmarks
Fit unigram/bigram vocabulary only on training data, with English stopword removal. Evaluate Naive Bayes and logistic regression on the same test set.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words='english')
X_train_vectors = vectorizer.fit_transform(X_train)
X_test_vectors = vectorizer.transform(X_test)

In [ ]:
model_nb = naive_bayes.MultinomialNB()
model_nb.fit(X_train_vectors, y_train)

In [ ]:

y_pred_nb = model_nb.predict(X_test_vectors)
print(classification_report(y_test, y_pred_nb, zero_division=0.0))

              precision    recall  f1-score   support

    negative       0.57      0.49      0.53        51
     neutral       0.80      0.88      0.84       297
    positive       0.67      0.55      0.61       137

    accuracy                           0.75       485
   macro avg       0.68      0.64      0.66       485
weighted avg       0.74      0.75      0.74       485



In [ ]:
model_lr = LogisticRegression()
model_lr.fit(X_train_vectors, y_train)

In [ ]:
y_pred_lr = model_lr.predict(X_test_vectors)
print(classification_report(y_test, y_pred_lr, zero_division=0.0))

              precision    recall  f1-score   support

    negative       0.65      0.39      0.49        51
     neutral       0.76      0.97      0.85       297
    positive       0.81      0.45      0.58       137

    accuracy                           0.76       485
   macro avg       0.74      0.60      0.64       485
weighted avg       0.76      0.76      0.74       485



## Off-the-shelf sentiment checkpoint
Load the checkpoint identifier used in the original analysis. It is a customer-comment sentiment model rather than a financial-domain model. Inference requires downloading model weights. Keep evaluation text local except for model downloads.

In [ ]:
import torch
from transformers import pipeline
if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
bert_path = 'MarieAngeA13/Sentiment-Analysis-BERT'
bert = pipeline(model=bert_path, device=device)


In [ ]:
y_pred_bert = [pred["label"] for pred in bert(X_test.to_list())]
print(classification_report(y_test, y_pred_bert, zero_division=0.0))

              precision    recall  f1-score   support

    negative       0.53      0.39      0.45        51
     neutral       0.68      0.90      0.77       297
    positive       0.57      0.21      0.31       137

    accuracy                           0.65       485
   macro avg       0.59      0.50      0.51       485
weighted avg       0.63      0.65      0.61       485



## Prepare supervised BERT adaptation
Map negative, neutral and positive to 0, 1 and 2; tokenize each split and wrap tensors in a PyTorch dataset. Before reproducing, inspect the checkpoint label mapping: overriding label metadata does not itself reorder pretrained classifier weights. Verify the original checkpoint head semantics against this explicit mapping, especially for the frozen-base comparison.

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

bert_path = "MarieAngeA13/Sentiment-Analysis-BERT"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_path)


In [ ]:
sentiment2int = {"negative": 0, "neutral": 1, "positive": 2}
int2sentiment = {value: key for key, value in sentiment2int.items()}

In [ ]:
def tokenize_and_tensorize(text, labels, tokenizer, label_mapping):
    encodings = tokenizer(
        text.tolist(),
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    label_tensor = torch.tensor(labels.map(label_mapping).tolist(), dtype=torch.long)
    return encodings, label_tensor


train_encodings, y_train_tensor = tokenize_and_tensorize(
    X_train, y_train, bert_tokenizer, sentiment2int
)
val_encodings, y_val_tensor = tokenize_and_tensorize(
    X_val, y_val, bert_tokenizer, sentiment2int
)
test_encodings, y_test_tensor = tokenize_and_tensorize(
    X_test, y_test, bert_tokenizer, sentiment2int
)


In [ ]:
class FinancialNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, index):
        item = {name: values[index] for name, values in self.encodings.items()}
        item["labels"] = self.labels[index]
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = FinancialNewsDataset(train_encodings, y_train_tensor)
val_dataset = FinancialNewsDataset(val_encodings, y_val_tensor)
test_dataset = FinancialNewsDataset(test_encodings, y_test_tensor)


## Head-only adaptation
Freeze the BERT base and train its classification head for ten epochs, with batch size 32. Evaluate and save each epoch. With no custom selection metric provided, best-checkpoint loading uses validation loss rather than test scores or macro F1. This selects among saved checkpoints; it does not guarantee absence of overfitting.

In [ ]:
model_frozen = AutoModelForSequenceClassification.from_pretrained(
    bert_path,
    num_labels=3,
    id2label=int2sentiment,
    label2id=sentiment2int,
    ignore_mismatched_sizes=True,
    attn_implementation="eager"
)

# Freeze the BERT base model so only the classification head trains
for param in model_frozen.bert.parameters():
    param.requires_grad = False

training_args_frozen = TrainingArguments(
    output_dir="./results_frozen",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_strategy="epoch",
    report_to="none"
)

trainer_frozen = Trainer(
    model=model_frozen,
    args=training_args_frozen,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer_frozen.train()

pred_output_frozen = trainer_frozen.predict(test_dataset)
y_pred_frozen = np.argmax(pred_output_frozen.predictions, axis=1)

print(classification_report(
    y_test_tensor.numpy(),
    y_pred_frozen,
    labels=list(int2sentiment.keys()),
    target_names=list(int2sentiment.values()),
    zero_division=0.0
))

              precision    recall  f1-score   support

    negative       0.52      0.59      0.55        51
     neutral       0.75      0.82      0.78       297
    positive       0.58      0.43      0.50       137

    accuracy                           0.68       485
   macro avg       0.62      0.61      0.61       485
weighted avg       0.68      0.68      0.68       485



## Full model fine-tuning
Initialize a separate copy and leave all parameters trainable. Use the same epoch count, batch size, validation evaluation and checkpoint selection procedure. GPU acceleration is recommended; checkpoint saving can use substantial disk space.

In [ ]:

model_unfrozen = AutoModelForSequenceClassification.from_pretrained(
    bert_path,
    num_labels=3,
    id2label=int2sentiment,
    label2id=sentiment2int,
    ignore_mismatched_sizes=True,
    attn_implementation="eager"
)

training_args_unfrozen = TrainingArguments(
    output_dir="./results_unfrozen",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_strategy="epoch",
    report_to="none"
)

trainer_unfrozen = Trainer(
    model=model_unfrozen,
    args=training_args_unfrozen,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:

trainer_unfrozen.train()

pred_output_unfrozen = trainer_unfrozen.predict(test_dataset)
y_pred_unfrozen = np.argmax(pred_output_unfrozen.predictions, axis=1)

print(classification_report(
    y_test_tensor.numpy(),
    y_pred_unfrozen,
    labels=list(int2sentiment.keys()),
    target_names=list(int2sentiment.values()),
    zero_division=0.0
))

              precision    recall  f1-score   support

    negative       0.76      0.86      0.81        51
     neutral       0.91      0.90      0.91       297
    positive       0.83      0.81      0.82       137

    accuracy                           0.87       485
   macro avg       0.83      0.86      0.84       485
weighted avg       0.87      0.87      0.87       485



## Main test findings
Full fine-tuning achieved reported accuracy 0.87, macro F1 0.84 and weighted F1 0.87. Logistic regression achieved 0.76 accuracy and 0.64 macro F1; head-only adaptation achieved 0.68 accuracy and 0.61 macro F1; the off-the-shelf BERT checkpoint achieved 0.65 accuracy and 0.51 macro F1. Naive Bayes reached 0.75 accuracy and 0.66 macro F1, exceeding logistic regression on the macro measure despite slightly lower accuracy.

Full fine-tuning's reported negative-class recall was 0.86 and positive-class recall was 0.81, compared with 0.39 and 0.45 for logistic regression. The rounded accuracy difference is approximately 11 percentage points. This is a single-split result, not a universal ranking of algorithms or proof of production performance.

## External FinBERT comparison: possible corpus overlap
ProsusAI/finbert is evaluated as a separate external model. The original analysis identifies Financial PhraseBank in its training history. Because the evaluation data comes from that corpus, prior exposure is possible; exact sentence overlap has not been audited here. Its reported 0.89 accuracy must not be treated as independent evidence of generalization or a clean win over locally trained models. Model information: https://huggingface.co/ProsusAI/finbert .

In [ ]:
finbert = pipeline(model="ProsusAI/finbert", device=device)


y_pred_finbert = [pred["label"] for pred in finbert(X_test.to_list())]
print(classification_report(y_test, y_pred_finbert, zero_division=0.0))


              precision    recall  f1-score   support

    negative       0.80      0.94      0.86        51
     neutral       0.96      0.87      0.92       297
    positive       0.80      0.91      0.85       137

    accuracy                           0.89       485
   macro avg       0.85      0.91      0.88       485
weighted avg       0.90      0.89      0.89       485



## Reproducibility and limitations
- Training and evaluation were not rerun; retained reports are rounded, and pretrained revisions and dependencies are unpinned.
- The original training used MPS and emitted pin-memory warnings; device availability and numerical behavior may differ.
- The test split is small and imbalanced, without deduplication or a temporal holdout. Repeated random splits and confidence intervals would strengthen conclusions.
- Audit checkpoint label ordering, specify seeds and model revisions, and record package versions before reproducing.
- Verify external-model training overlap and evaluate on newly collected, independently labeled financial news before deployment.

## Author
Cameron Batts